### A. Connection Establishment:

In [1]:
import duckdb
import pandas as pd

# 1. Establish the connection
db_path = r'B:\3. Prog\2. Projects\7. Logistics and supply chain\logistics.db'
con = duckdb.connect(db_path)

# 2. Helper function
def sql(query_string):
    return con.execute(query_string).df()

print("--- 🛰️ LOGISTICS DB CONNECTED ---")

# 3. AUTO-DETECT TABLE NAMES
# This pulls the actual names from your DB catalog
tables = con.execute("SHOW TABLES").df()['name'].tolist()
print(f"Detected Tables in DB: {tables}")

# Determine which names are being used (Singular vs Plural)
f_table = 'fact_logistics' if 'fact_logistics' in tables else 'fact_shipments'
d_table = 'dim_driver' if 'dim_driver' in tables else 'dim_drivers'

if f_table in tables and d_table in tables:
    # 4. RUN THE AUDIT WITH DETECTED NAMES
    red_flag_query = f"""
    SELECT 
        f.Shipment_ID, 
        f.driver_id, 
        d.driver_behavior_score,
        f.shipping_costs
    FROM {f_table} f
    LEFT JOIN {d_table} d ON f.driver_id = d.driver_id
    WHERE d.driver_behavior_score IS NULL
    LIMIT 10;
    """

    print(f"\n[Red Flag Investigation] Using tables: {f_table} + {d_table}")
    results = sql(red_flag_query)
    
    if results.empty:
        print("No missing driver scores found in this sample.")
    else:
        print("Shipments with NULL Driver Scores Found:")
        display(results)
else:
    print(f"❌ Error: Could not find necessary tables. Found: {tables}")

--- 🛰️ LOGISTICS DB CONNECTED ---
Detected Tables in DB: ['dim_drivers', 'dim_routes', 'dim_suppliers', 'dim_vehicles', 'fact_shipments']

[Red Flag Investigation] Using tables: fact_shipments + dim_drivers
Shipments with NULL Driver Scores Found:


,Shipment_ID,driver_id,driver_behavior_score,shipping_costs
0,1361292,DRV_10285,NaN,771.145397
1,1361314,DRV_10288,NaN,939.674791
2,1361319,DRV_10290,NaN,266.511627
3,1361330,DRV_10295,NaN,110.947464
4,1361395,DRV_10293,NaN,675.858786
5,1361407,DRV_10295,NaN,116.289408
6,1361425,DRV_10288,NaN,135.947344
7,1361432,DRV_10290,NaN,998.006621
8,1361444,DRV_10294,NaN,996.165105
9,1361446,DRV_10294,NaN,693.833803


### B. Table Names:

In [2]:
# Fetch all table names into a list
tables = con.execute("SHOW TABLES").df()['name'].tolist()

print(f"--- 🛰️ CONNECTION RE-ESTABLISHED ---")

if tables:
    # Loop through the list to confirm each table individually
    for table in tables:
        print(f"✅ Table Ready: {table}")
else:
    print("⚠️ No tables found in the current connection.")

# Keeping your f_table logic for subsequent KPI queries
f_table = 'fact_logistics' if 'fact_logistics' in tables else 'fact_shipments'

--- 🛰️ CONNECTION RE-ESTABLISHED ---
✅ Table Ready: dim_drivers
✅ Table Ready: dim_routes
✅ Table Ready: dim_suppliers
✅ Table Ready: dim_vehicles
✅ Table Ready: fact_shipments


In [3]:
# Check for the table containing temperature/sensor data
print(con.execute("SHOW TABLES").df())

             name
0     dim_drivers
1      dim_routes
2   dim_suppliers
3    dim_vehicles
4  fact_shipments


In [4]:
# Checking columns in dim_vehicles and dim_routes to find the temperature sensor data
print("Columns in dim_vehicles:", con.execute("PRAGMA table_info('dim_vehicles')").df()['name'].tolist())
print("Columns in dim_routes:", con.execute("PRAGMA table_info('dim_routes')").df()['name'].tolist())
print("Columns in dim_drivers:", con.execute("PRAGMA table_info('dim_drivers')").df()['name'].tolist())
print("Columns in dim_suppliers:", con.execute("PRAGMA table_info('dim_suppliers')").df()['name'].tolist())
print("Columns in fact_shipments:", con.execute("PRAGMA table_info('fact_shipments')").df()['name'].tolist())

Columns in dim_vehicles: ['vehicle_id', 'fuel_consumption_rate', 'loading_unloading_time', 'risk_classification']
Columns in dim_routes: ['route_id', 'vehicle_gps_latitude', 'vehicle_gps_longitude', 'route_risk_level']
Columns in dim_drivers: ['driver_id', 'driver_behavior_score', 'fatigue_monitoring_score']
Columns in dim_suppliers: ['supplier_id', 'supplier_reliability_score', 'port_congestion_level']
Columns in fact_shipments: ['Shipment_ID', 'timestamp', 'driver_id', 'vehicle_id', 'route_id', 'supplier_id', 'shipping_costs', 'delivery_time_deviation', 'iot_temperature', 'order_fulfillment_status']


In [45]:
# 1. Fetch all table names
tables = con.execute("SHOW TABLES").df()['name'].tolist()

print(f"--- 📊 DATABASE SCHEMA OVERVIEW (SHAPES) ---")

for table in tables:
    # Get row count
    row_count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    
    # Get column count by querying the information_schema
    col_count = con.execute(f"""
        SELECT COUNT(*) 
        FROM information_schema.columns 
        WHERE table_name = '{table}'
    """).fetchone()[0]
    
    # Format and print the 'Shape' similar to df.shape
    print(f"📦 Table: {table:<20} | Shape: ({row_count}, {col_count})")

print(f"--------------------------------------------")

--- 📊 DATABASE SCHEMA OVERVIEW (SHAPES) ---
📦 Table: dim_drivers          | Shape: (10298, 3)
📦 Table: dim_routes           | Shape: (1114, 4)
📦 Table: dim_suppliers        | Shape: (1123, 3)
📦 Table: dim_vehicles         | Shape: (21008, 4)
📦 Table: fact_shipments       | Shape: (4996793, 10)
--------------------------------------------


In [6]:
import pandas as pd

# 1. Get the list of all tables in the connection
tables = con.execute("SHOW TABLES").df()['name'].tolist()

for table in tables:
    print(f"\n--- 📋 COLUMN SUMMARY: {table.upper()} ---")
    
    # 2. Fetch basic metadata (Name, Type, Nullability) from information_schema
    metadata_query = f"""
    SELECT 
        column_name AS "Column",
        data_type AS "Type",
        is_nullable AS "Is_Nullable"
    FROM information_schema.columns 
    WHERE table_name = '{table}'
    ORDER BY ordinal_position
    """
    metadata = con.execute(metadata_query).df()
    
    # 3. Dynamically fetch Profiling Metrics (Nulls and Uniques) for each column
    profiling_list = []
    for col in metadata['Column']:
        # We use double quotes around {col} to handle spaces or reserved words
        profile_query = f"""
        SELECT 
            '{col}' AS "Column",
            COUNT(*) - COUNT("{col}") AS "Null_Count",
            COUNT(DISTINCT "{col}") AS "Unique_Values"
        FROM {table}
        """
        profiling_list.append(con.execute(profile_query).df())
    
    # 4. Merge metadata with profiling results
    profiling_df = pd.concat(profiling_list)
    final_summary = metadata.merge(profiling_df, on="Column")
    
    # 5. Display the final formatted summary
    print(final_summary.to_string(index=False))
    print("-" * 60)


--- 📋 COLUMN SUMMARY: DIM_DRIVERS ---
                  Column    Type Is_Nullable  Null_Count  Unique_Values
               driver_id VARCHAR         YES           0          10298
   driver_behavior_score  DOUBLE         YES          14          10252
fatigue_monitoring_score  DOUBLE         YES           1          10257
------------------------------------------------------------

--- 📋 COLUMN SUMMARY: DIM_ROUTES ---
               Column    Type Is_Nullable  Null_Count  Unique_Values
             route_id VARCHAR         YES           0           1114
 vehicle_gps_latitude  DOUBLE         YES           1           1113
vehicle_gps_longitude  DOUBLE         YES           1           1113
     route_risk_level  DOUBLE         YES           1           1111
------------------------------------------------------------

--- 📋 COLUMN SUMMARY: DIM_SUPPLIERS ---
                    Column    Type Is_Nullable  Null_Count  Unique_Values
               supplier_id VARCHAR         YES       

### Phase 1: Basic Operational Health 

In [7]:
# A. Operational Efficiency
# i> Total Shipment
# Purpose: This establishes the scale of the entire logistics network.

# Using your established f_table variable from the previous cell
total_shipment_query = f"""
SELECT 
    COUNT(*) AS total_shipments 
FROM {f_table};
"""

print(f"--- 📊 KPI: OPERATIONAL EFFICIENCY ---")
print(f"Target Table: {f_table}")

# Execute using your sql() helper function
total_results = sql(total_shipment_query)

# Display the results
display(total_results)

--- 📊 KPI: OPERATIONAL EFFICIENCY ---
Target Table: fact_shipments


,total_shipments
0,4996793


In [8]:
# A. Operational Efficiency
# ii> Fulfillment Status Distribution
# Purpose: Shows the health of the shipping pipeline and identifies volume bottlenecks.

status_dist_query = f"""
SELECT 
    order_fulfillment_status, 
    COUNT(*) AS shipment_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM {f_table}), 2) AS percentage
FROM {f_table}
GROUP BY order_fulfillment_status
ORDER BY shipment_count DESC;
"""

print(f"--- 📊 KPI: STATUS DISTRIBUTION ---")
status_results = sql(status_dist_query)

# Display the results
display(status_results)

--- 📊 KPI: STATUS DISTRIBUTION ---


,order_fulfillment_status,shipment_count,percentage
0,0.000000,33483,0.67
1,NaN,4792,0.10
2,0.865564,1,0.00
3,0.996581,1,0.00
4,1.001062,1,0.00
...,...,...,...
4958515,0.877933,1,0.00
4958516,0.465989,1,0.00
4958517,0.997308,1,0.00
4958518,0.996837,1,0.00


In [33]:
# A. Operational Efficiency
# iii> Volume by Shift
# Purpose: Identifies peak operational periods to optimize staffing and monitoring.
# Logic: Categorizes 24 hours into Morning, Afternoon, and Night (US/Rotational) shifts.

volume_shift_query = f"""
SELECT 
    CASE 
        WHEN HOUR(timestamp) BETWEEN 6 AND 13 THEN 'Morning Shift'
        WHEN HOUR(timestamp) BETWEEN 14 AND 21 THEN 'Afternoon Shift'
        ELSE 'Night Shift (US Shift)'
    END AS shift_name,
    COUNT(*) AS shipment_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM {f_table}), 2) AS volume_percentage
FROM {f_table}
GROUP BY shift_name
ORDER BY shipment_count DESC;
"""

print(f"--- 🕒 BEGINNER KPI: VOLUME BY SHIFT ---")
shift_results = sql(volume_shift_query)

# Display results
display(shift_results)

--- 🕒 BEGINNER KPI: VOLUME BY SHIFT ---


,shift_name,shipment_count,volume_percentage
0,Morning Shift,1666557,33.35
1,Afternoon Shift,1665506,33.33
2,Night Shift (US Shift),1664730,33.32


In [34]:
# A. Operational Efficiency
# iv> Asset Utilization Rate
# Purpose: Measures the efficiency of fleet deployment.
# Logic: (Unique Vehicles in Fact / Total Vehicles in Dimension) * 100

asset_utilization_query = """
SELECT 
    (SELECT COUNT(DISTINCT vehicle_id) FROM fact_shipments) AS active_vehicles,
    (SELECT COUNT(*) FROM dim_vehicles) AS total_fleet_size,
    ROUND(100.0 * (SELECT COUNT(DISTINCT vehicle_id) FROM fact_shipments) / 
          (SELECT COUNT(*) FROM dim_vehicles), 2) AS utilization_rate_pct
"""

print(f"--- 🚛 BEGINNER KPI: ASSET UTILIZATION RATE ---")
# Assuming the sql() helper is connected to your duckdb instance
utilization_results = sql(asset_utilization_query)

# Display results
display(utilization_results)

--- 🚛 BEGINNER KPI: ASSET UTILIZATION RATE ---


,active_vehicles,total_fleet_size,utilization_rate_pct
0,21008,21008,100.0


In [ ]:
# B. Financial & Cost Management
# i> Total Logistics Spend by Supplier
# Purpose: Identifies high-value vendors and total financial throughput per supplier.

total_spend_query = f"""
SELECT 
    supplier_id, 
    ROUND(SUM(shipping_costs), 2) AS total_spend,
    COUNT(*) AS shipment_count,
    ROUND(AVG(shipping_costs), 2) AS avg_cost_per_shipment
FROM {f_table}
GROUP BY supplier_id
ORDER BY total_spend DESC;
"""

print(f"--- 💰 KPI: FINANCIAL & COST MANAGEMENT ---")
spend_results = sql(total_spend_query)

# Display the results
display(spend_results) 

--- 💰 KPI: FINANCIAL & COST MANAGEMENT ---


,supplier_id,total_spend,shipment_count,avg_cost_per_shipment
0,SUP_00101,28064080.15,59065,477.63
1,SUP_01121,25153817.40,55719,461.11
2,SUP_01019,20668257.05,45593,460.96
3,SUP_01120,18402000.66,39997,464.67
4,SUP_00203,18074844.23,41253,442.38
...,...,...,...,...
1118,SUP_00817,104322.67,200,521.61
1119,SUP_00715,90337.31,211,428.14
1120,SUP_00307,83780.16,162,558.53
1121,SUP_00613,82805.81,185,447.60


In [11]:
# B. Financial & Cost Management
# ii> Average Shipping Cost per Route
# Purpose: Identifies high-cost corridors and geographical spend distribution.

supplier_avg_cost_query = f"""
SELECT 
    supplier_id, 
    COUNT(*) AS shipment_volume,
    ROUND(AVG(shipping_costs), 2) AS avg_supplier_cost,
    MIN(shipping_costs) AS min_cost,
    MAX(shipping_costs) AS max_cost
FROM {f_table}
GROUP BY supplier_id
ORDER BY avg_supplier_cost DESC;
"""

print(f"--- 🚚 KPI: SUPPLIER COST ANALYSIS ---")
supplier_results = sql(supplier_avg_cost_query)

# Display the results
display(supplier_results)

--- 🚚 KPI: SUPPLIER COST ANALYSIS ---


,supplier_id,shipment_volume,avg_supplier_cost,min_cost,max_cost
0,SUP_00484,2926,682.52,106.365037,1006.976515
1,SUP_00858,1604,660.78,94.342239,1007.633696
2,SUP_00583,2074,633.84,93.021085,980.636144
3,SUP_00546,1336,624.43,93.234097,1009.548077
4,SUP_00146,3643,614.66,90.877882,1004.948300
...,...,...,...,...,...
1118,SUP_00365,2079,317.96,91.593947,990.735430
1119,SUP_00516,3319,307.89,91.214807,1004.111740
1120,SUP_00882,2297,293.68,91.933572,1001.123335
1121,SUP_00439,1833,267.27,90.882806,986.596843


In [16]:
# C. Quality Control & Compliance
# i> Critical Temperature Breach
# Purpose: Quantifies the volume of product safety violations.

temp_breach_query = f"""
SELECT 
    COUNT(*) AS total_shipments,
    SUM(CASE WHEN iot_temperature > 15 THEN 1 ELSE 0 END) AS critical_breaches,
    ROUND(100.0 * SUM(CASE WHEN iot_temperature > 15 THEN 1 ELSE 0 END) / COUNT(*), 2) AS breach_rate_percentage
FROM {f_table};
"""

print(f"--- 🌡️ KPI: THERMAL COMPLIANCE AUDIT ---")
display(sql(temp_breach_query))

--- 🌡️ KPI: THERMAL COMPLIANCE AUDIT ---


,total_shipments,critical_breaches,breach_rate_percentage
0,4996793,888075.0,17.77


In [17]:
# D. Supplier & Carrier Performance
# i> Load Distribution
# Purpose: Identifies workload balance and potential over-reliance on specific vendors.

load_dist_query = f"""
SELECT 
    supplier_id, 
    COUNT(*) AS total_shipments,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM {f_table}), 2) AS volume_percentage
FROM {f_table}
GROUP BY supplier_id
ORDER BY total_shipments DESC
LIMIT 20;
"""

print(f"--- 📦 KPI: SUPPLIER LOAD DISTRIBUTION ---")
load_results = sql(load_dist_query)

# Display the results
display(load_results)

--- 📦 KPI: SUPPLIER LOAD DISTRIBUTION ---


,supplier_id,total_shipments,volume_percentage
0,SUP_00101,59065,1.18
1,SUP_01121,55719,1.12
2,SUP_01019,45593,0.91
3,SUP_00203,41253,0.83
4,SUP_01120,39997,0.80
5,SUP_00100,39089,0.78
6,SUP_00917,32208,0.64
7,SUP_01018,30763,0.62
8,SUP_00305,30608,0.61
9,SUP_00202,30172,0.60


### Phase 2: Performance & Compliance Audit

In [18]:
# A. Operational Efficiency
# i> Average Delivery Time
# Purpose: Quantifies the accuracy of delivery schedules. 
# Positive values = Late, Negative values = Early, 0 = On Time.

delivery_time_query = f"""
SELECT 
    AVG(delivery_time_deviation) AS global_avg_deviation,
    MIN(delivery_time_deviation) AS max_early_delivery,
    MAX(delivery_time_deviation) AS max_late_delivery,
    -- Using CASE to categorize performance
    COUNT(CASE WHEN delivery_time_deviation > 0 THEN 1 END) AS late_shipments,
    COUNT(CASE WHEN delivery_time_deviation <= 0 THEN 1 END) AS on_time_or_early
FROM {f_table};
"""

print(f"--- 🛰️ INTERMEDIATE KPI: DELIVERY PERFORMANCE ---")
delivery_results = sql(delivery_time_query)

# Display the results
display(delivery_results)

--- 🛰️ INTERMEDIATE KPI: DELIVERY PERFORMANCE ---


,global_avg_deviation,max_early_delivery,max_late_delivery,late_shipments,on_time_or_early
0,5.09965,-10.868152,10.183809,4077651,914350


In [19]:
# A. Operational Efficiency
# ii> Route Load-Time Analysis
# Purpose: Compares delivery delays across different route risk categories.
# Demonstrates: SQL JOINs and Multi-table Aggregation.

route_lead_time_query = f"""
SELECT 
    r.route_risk_level,
    COUNT(f.Shipment_ID) AS total_shipments,
    ROUND(AVG(f.delivery_time_deviation), 2) AS avg_lead_time_deviation,
    MAX(f.delivery_time_deviation) AS extreme_delay,
    MIN(f.delivery_time_deviation) AS best_case_lead_time
FROM {f_table} f
JOIN dim_routes r ON f.route_id = r.route_id
GROUP BY r.route_risk_level
ORDER BY avg_lead_time_deviation DESC;
"""

print(f"--- 🛣️ INTERMEDIATE KPI: ROUTE LEAD-TIME BY RISK ---")
route_lead_results = sql(route_lead_time_query)

# Display the results
display(route_lead_results)

--- 🛣️ INTERMEDIATE KPI: ROUTE LEAD-TIME BY RISK ---


,route_risk_level,total_shipments,avg_lead_time_deviation,extreme_delay,best_case_lead_time
0,10.001039,3,9.28,9.959538,8.113888
1,8.063240,1907,7.63,10.107344,-10.868152
2,7.555084,2135,7.44,10.101100,-10.868152
3,9.652195,1635,7.42,10.070370,-10.868152
4,2.855685,1714,7.34,10.113069,-10.868152
...,...,...,...,...,...
1107,1.039455,4,-0.26,4.444553,-10.868152
1108,4.053753,1,-0.33,-0.326858,-0.326858
1109,9.993748,2,-1.56,-1.532328,-1.580645
1110,4.461224,1,-1.99,-1.986608,-1.986608


In [20]:
# B. Financial & Cost Management
# i> cost-to-Weight Efficiency
# Purpose: Measures the cost of transport per unit of weight to identify shipping inefficiency.

# NOTE: If 'weight_kg' is not in your table, this query demonstrates the JOIN logic
# required once that data is ingested.

cost_weight_query = f"""
SELECT 
    f.supplier_id,
    ROUND(SUM(f.shipping_costs), 2) AS total_cost,
    -- We are calculating a derived 'Efficiency Score'
    -- If weight is unavailable, we use Shipment Count as a proxy for 'Unit Cost'
    ROUND(SUM(f.shipping_costs) / COUNT(f.Shipment_ID), 2) AS cost_per_unit,
    
    -- Using CASE to flag inefficient shipments (High Cost, Low Volume/Weight)
    COUNT(CASE WHEN f.shipping_costs > 900 THEN 1 END) AS high_cost_shipments
FROM {f_table} f
GROUP BY f.supplier_id
ORDER BY cost_per_unit DESC
LIMIT 15;
"""

print(f"--- 💸 INTERMEDIATE KPI: COST EFFICIENCY ---")
cost_results = sql(cost_weight_query)

# Display the results
display(cost_results)

--- 💸 INTERMEDIATE KPI: COST EFFICIENCY ---


,supplier_id,total_cost,cost_per_unit,high_cost_shipments
0,SUP_00858,1045353.95,651.72,691
1,SUP_00583,1314580.82,633.84,302
2,SUP_00484,1853037.59,633.30,912
3,SUP_00546,832992.35,623.50,426
4,SUP_00278,1660432.64,609.33,733
5,SUP_00664,1310210.94,607.70,604
6,SUP_00562,790354.31,600.12,212
7,SUP_00349,866335.41,596.24,243
8,SUP_00773,694717.29,593.78,166
9,SUP_00262,1635501.75,590.65,954


In [21]:
# C. Quality Control & Compliance
# i> Thermal Compliance Rate
# Purpose: Measures the percentage of shipments that maintained required temperature standards.
# Threshold: <= 15°C is Compliant | > 15°C is a Breach.

thermal_compliance_query = f"""
WITH ThermalStats AS (
    SELECT 
        CASE 
            WHEN iot_temperature <= 15 THEN 'Compliant'
            ELSE 'Breach'
        END AS thermal_status
    FROM {f_table}
)
SELECT 
    thermal_status,
    COUNT(*) AS shipment_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM ThermalStats), 2) AS compliance_rate
FROM ThermalStats
GROUP BY thermal_status;
"""

print(f"--- 🌡️ INTERMEDIATE KPI: THERMAL COMPLIANCE RATE ---")
thermal_results = sql(thermal_compliance_query)

# Display results
display(thermal_results)

--- 🌡️ INTERMEDIATE KPI: THERMAL COMPLIANCE RATE ---


,thermal_status,shipment_count,compliance_rate
0,Breach,892867,17.87
1,Compliant,4103926,82.13


In [22]:
# C. Quality Control & Compliance
# ii> Average Temperature Stability
# Purpose: Measures the consistency of the cold chain environment.
# Logic: Lower Standard Deviation = Higher Stability.

temp_stability_query = f"""
SELECT 
    route_id,
    ROUND(AVG(iot_temperature), 2) AS avg_temp,
    ROUND(STDDEV(iot_temperature), 2) AS temp_volatility, -- Standard Deviation
    COUNT(*) AS reading_count,
    MAX(iot_temperature) - MIN(iot_temperature) AS temp_range
FROM {f_table}
GROUP BY route_id
HAVING reading_count > 10 -- Ensuring statistical significance
ORDER BY temp_volatility DESC
LIMIT 15;
"""

print(f"--- 🌡️ INTERMEDIATE KPI: TEMPERATURE STABILITY ---")
stability_results = sql(temp_stability_query)

# Display results
display(stability_results)

--- 🌡️ INTERMEDIATE KPI: TEMPERATURE STABILITY ---


,route_id,avg_temp,temp_volatility,reading_count,temp_range
0,RTE_01007,7.02,9.21,13,18.600069
1,RTE_00727,8.08,9.14,1320,18.600069
2,RTE_00911,8.05,9.07,1972,18.600069
3,RTE_00507,10.01,9.03,1673,18.600069
4,RTE_00514,7.48,8.91,2695,18.600069
5,RTE_00708,7.03,8.88,1202,18.600069
6,RTE_00553,6.66,8.86,1314,18.600069
7,RTE_00284,7.50,8.84,1907,18.600069
8,RTE_00392,6.77,8.83,2530,18.600069
9,RTE_00438,8.22,8.82,1539,18.600069


In [23]:
# D. Supplier & Carrier Performance
# i> Carrier Delay Frequency
# Purpose: Identifies which suppliers have the highest frequency of late deliveries.
# Logic: (Total Late Shipments / Total Shipments per Supplier) * 100

carrier_delay_query = f"""
SELECT 
    supplier_id,
    COUNT(*) AS total_shipments,
    COUNT(CASE WHEN delivery_time_deviation > 0 THEN 1 END) AS total_delayed_shipments,
    ROUND(100.0 * COUNT(CASE WHEN delivery_time_deviation > 0 THEN 1 END) / COUNT(*), 2) AS delay_frequency_pct,
    ROUND(AVG(delivery_time_deviation), 2) AS avg_delay_magnitude
FROM {f_table}
GROUP BY supplier_id
HAVING total_shipments > 50 -- Filter for statistical significance
ORDER BY delay_frequency_pct DESC
LIMIT 15;
"""

print(f"--- 🚚 ADVANCED KPI: CARRIER DELAY FREQUENCY ---")
carrier_results = sql(carrier_delay_query)

# Display results
display(carrier_results)

--- 🚚 ADVANCED KPI: CARRIER DELAY FREQUENCY ---


,supplier_id,total_shipments,total_delayed_shipments,delay_frequency_pct,avg_delay_magnitude
0,SUP_00726,3835,3813,99.43,6.98
1,SUP_00548,1350,1339,99.19,7.50
2,SUP_00750,1631,1606,98.47,6.66
3,SUP_00770,1729,1690,97.74,6.86
4,SUP_00435,2285,2230,97.59,6.08
5,SUP_00875,2421,2350,97.07,7.26
6,SUP_00027,4777,4629,96.90,7.31
7,SUP_00766,2182,2109,96.65,6.57
8,SUP_00727,3413,3298,96.63,5.45
9,SUP_00248,1731,1672,96.59,5.56


In [24]:
# D. Supplier & Carrier Performance
# ii> Vendor Thermal Integrity
# Purpose: Ranks suppliers by their ability to maintain temperature compliance.
# Integrity Score: % of shipments that stayed BELOW or AT 15°C.

vendor_thermal_query = f"""
WITH VendorCompliance AS (
    SELECT 
        supplier_id,
        COUNT(*) AS total_shipments,
        SUM(CASE WHEN iot_temperature <= 15 THEN 1 ELSE 0 END) AS compliant_count
    FROM {f_table}
    GROUP BY supplier_id
)
SELECT 
    supplier_id,
    total_shipments,
    compliant_count,
    ROUND(100.0 * compliant_count / total_shipments, 2) AS integrity_score_pct
FROM VendorCompliance
WHERE total_shipments > 100 -- Focus on vendors with significant volume
ORDER BY integrity_score_pct ASC -- Show worst performers (Red Flags) first
LIMIT 15;
"""

print(f"--- 🌡️ ADVANCED KPI: VENDOR THERMAL INTEGRITY ---")
vendor_thermal_results = sql(vendor_thermal_query)

# Display results
display(vendor_thermal_results)

--- 🌡️ ADVANCED KPI: VENDOR THERMAL INTEGRITY ---


,supplier_id,total_shipments,compliant_count,integrity_score_pct
0,SUP_01123,4792,0.0,0.00
1,SUP_00538,2577,1453.0,56.38
2,SUP_00634,1670,1003.0,60.06
3,SUP_00789,2650,1607.0,60.64
4,SUP_00569,1773,1081.0,60.97
5,SUP_00539,1613,984.0,61.00
6,SUP_00574,899,557.0,61.96
7,SUP_00781,1730,1079.0,62.37
8,SUP_00497,3817,2421.0,63.43
9,SUP_00750,1631,1048.0,64.26


### Phase 3: Strategic Insights & Benchmarking

In [25]:
# A. Operational Efficiency
# i> Bottleneck Identification
# Purpose: Highlights specific routes where high volume meets high delay.
# Logic: (Total Shipments * Avg Deviation) = Impact Score.

bottleneck_query = f"""
SELECT 
    route_id,
    COUNT(*) AS shipment_volume,
    ROUND(AVG(delivery_time_deviation), 2) AS avg_delay,
    ROUND(COUNT(*) * AVG(delivery_time_deviation), 2) AS bottleneck_impact_score
FROM {f_table}
GROUP BY route_id
HAVING shipment_volume > 100 AND avg_delay > 0
ORDER BY bottleneck_impact_score DESC
LIMIT 10;
"""

print(f"--- 🚧 ADVANCED KPI: BOTTLENECK IDENTIFICATION ---")
bottleneck_results = sql(bottleneck_query)

# Display results
display(bottleneck_results)

--- 🚧 ADVANCED KPI: BOTTLENECK IDENTIFICATION ---


,route_id,shipment_volume,avg_delay,bottleneck_impact_score
0,RTE_00052,96493,5.13,495309.95
1,RTE_00105,60355,5.20,314139.46
2,RTE_00051,56526,5.09,287462.73
3,RTE_00158,39819,5.10,203180.23
4,RTE_00050,37511,5.29,198318.71
5,RTE_01112,35540,4.96,176448.64
6,RTE_01059,33142,4.98,164957.95
7,RTE_00104,31243,5.06,158021.16
8,RTE_00002,29545,5.00,147837.11
9,RTE_00048,26699,5.48,146180.76


In [ ]:
# A. Operational Efficiency
# ii> Route Circuitry (Planned vs. Actual)
# Purpose: Identifies routes where actual movement deviates from planned efficiency.
# Logic: High ETA Variation + High Route Risk = Circuitry Issue (Detours/Inefficiency).

query = """
SELECT 
    ROUND(vehicle_gps_latitude, 1) AS lat_zone,
    ROUND(vehicle_gps_longitude, 1) AS lon_zone,
    COUNT(*) AS trip_count,
    ROUND(AVG(eta_variation_hours), 2) AS avg_planned_var_hrs,
    ROUND(AVG(delivery_time_deviation), 2) AS avg_actual_delay_mins,
    ROUND(AVG(eta_variation_hours * delivery_time_deviation), 2) AS circuitry_impact_score
FROM df
GROUP BY ROUND(vehicle_gps_latitude, 1), ROUND(vehicle_gps_longitude, 1)
HAVING COUNT(*) > 50
ORDER BY circuitry_impact_score DESC
LIMIT 10;
"""

circuitry_df = con.execute(query).df()
display(circuitry_df)

,lat_zone,lon_zone,trip_count,avg_planned_var_hrs,avg_actual_delay_mins,circuitry_impact_score
0,30.1,-70.0,53,3.03,5.86,17.03
1,30.0,-70.0,92,2.85,4.73,11.96


In [26]:
# B. Financial & Cost Management
# i> Cost-to-Carrier Variance
# Purpose: Identifies carriers charging significantly above the network average.
# Logic: Positive Variance = Overpaying | Negative Variance = Cost Efficient.

cost_variance_query = f"""
WITH CarrierAverages AS (
    SELECT 
        supplier_id,
        COUNT(*) AS shipment_count,
        AVG(shipping_costs) AS carrier_avg_cost,
        -- Window function to get the global average across all shipments
        AVG(AVG(shipping_costs)) OVER() AS global_benchmark_cost
    FROM {f_table}
    GROUP BY supplier_id
    HAVING shipment_count > 100
)
SELECT 
    supplier_id,
    shipment_count,
    ROUND(carrier_avg_cost, 2) AS carrier_avg,
    ROUND(global_benchmark_cost, 2) AS benchmark,
    ROUND(carrier_avg_cost - global_benchmark_cost, 2) AS cost_variance,
    ROUND(((carrier_avg_cost - global_benchmark_cost) / global_benchmark_cost) * 100, 2) AS variance_percentage
FROM CarrierAverages
ORDER BY cost_variance DESC
LIMIT 15;
"""

print(f"--- 💸 ADVANCED KPI: COST-TO-CARRIER VARIANCE ---")
variance_results = sql(cost_variance_query)

# Display results
display(variance_results)

--- 💸 ADVANCED KPI: COST-TO-CARRIER VARIANCE ---


,supplier_id,shipment_count,carrier_avg,benchmark,cost_variance,variance_percentage
0,SUP_00484,2926,682.52,462.3,220.22,47.64
1,SUP_00858,1604,660.78,462.3,198.48,42.93
2,SUP_00583,2074,633.84,462.3,171.54,37.11
3,SUP_00546,1336,624.43,462.3,162.14,35.07
4,SUP_00146,3643,614.66,462.3,152.37,32.96
5,SUP_00278,2725,613.84,462.3,151.54,32.78
6,SUP_00664,2156,607.70,462.3,145.41,31.45
7,SUP_00578,2025,604.66,462.3,142.36,30.79
8,SUP_00562,1317,600.57,462.3,138.28,29.91
9,SUP_00349,1453,596.24,462.3,133.94,28.97


In [27]:
# B. Financial & Cost Management
# ii> Rolling Spend Trends
# Purpose: Smooths out daily cost fluctuations to identify long-term spending patterns.
# Logic: Calculates the average spend of the current day + the previous 6 days.

rolling_spend_query = f"""
WITH DailySpend AS (
    SELECT 
        CAST(timestamp AS DATE) AS shipment_date,
        SUM(shipping_costs) AS daily_total
    FROM {f_table}
    GROUP BY 1
)
SELECT 
    shipment_date,
    ROUND(daily_total, 2) AS daily_total,
    ROUND(AVG(daily_total) OVER (
        ORDER BY shipment_date 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS rolling_7day_avg
FROM DailySpend
ORDER BY shipment_date DESC
LIMIT 30;
"""

print(f"--- 📈 ADVANCED KPI: ROLLING SPEND TRENDS ---")
rolling_results = sql(rolling_spend_query)

# Display results
display(rolling_results)

--- 📈 ADVANCED KPI: ROLLING SPEND TRENDS ---


,shipment_date,daily_total,rolling_7day_avg
0,2028-09-07,608.18,29263.30
1,2028-09-06,40204.57,36025.03
2,2028-09-05,31841.73,34866.32
3,2028-09-04,29680.14,35219.51
4,2028-09-03,23868.54,35771.22
5,2028-09-02,42219.81,37070.27
6,2028-09-01,36420.11,35953.07
7,2028-08-31,47940.32,35381.39
8,2028-08-30,32093.56,33083.84
9,2028-08-29,34314.11,34786.72


In [28]:
# C. Quality Control & Compliance
# i> Condition based Risk Score
# Purpose: Assigns a 0-100 risk value to each shipment based on multiple failure points.
# Logic: Composite Score = (Thermal_Risk + Delay_Risk + Cost_Risk)

risk_score_query = f"""
WITH RiskFactorCalculation AS (
    SELECT 
        Shipment_ID,
        supplier_id,
        route_id,
        -- Weighted Logic for Risk Factors
        CASE WHEN iot_temperature > 15 THEN 40 ELSE 0 END AS thermal_risk_points,
        CASE WHEN delivery_time_deviation > 30 THEN 30 ELSE 0 END AS delay_risk_points,
        CASE WHEN shipping_costs > 950 THEN 30 ELSE 0 END AS cost_risk_points
    FROM {f_table}
)
SELECT 
    supplier_id,
    route_id,
    COUNT(*) AS total_shipments,
    ROUND(AVG(thermal_risk_points + delay_risk_points + cost_risk_points), 2) AS avg_risk_index,
    MAX(thermal_risk_points + delay_risk_points + cost_risk_points) AS max_risk_observed
FROM RiskFactorCalculation
GROUP BY supplier_id, route_id
HAVING total_shipments > 50
ORDER BY avg_risk_index DESC
LIMIT 15;
"""

print(f"--- 🚩 ADVANCED KPI: CONDITION-BASED RISK SCORE ---")
risk_results = sql(risk_score_query)

# Display results
display(risk_results)

--- 🚩 ADVANCED KPI: CONDITION-BASED RISK SCORE ---


,supplier_id,route_id,total_shipments,avg_risk_index,max_risk_observed
0,SUP_00802,RTE_00366,55,70.0,70
1,SUP_01118,RTE_00786,84,70.0,70
2,SUP_00370,RTE_00089,141,70.0,70
3,SUP_00995,RTE_00680,116,70.0,70
4,SUP_01116,RTE_00155,60,70.0,70
5,SUP_00282,RTE_00882,60,70.0,70
6,SUP_00812,RTE_00272,71,70.0,70
7,SUP_00822,RTE_00809,139,70.0,70
8,SUP_00099,RTE_00414,88,70.0,70
9,SUP_00877,RTE_00679,96,70.0,70


In [31]:
# C. Quality Control & Compliance
# ii> Carrier "First-Time-Right" (FTR) Rate
# Purpose: Measures absolute operational perfection per supplier.
# Logic: Shipment is 'Right' ONLY if Delay=0, Temp<=15, and Cost is not an Anomaly.

ftr_query = f"""
SELECT 
    supplier_id,
    COUNT(*) AS total_shipments,
    SUM(CASE 
        WHEN delivery_time_deviation <= 0 
        AND iot_temperature <= 15 
        AND shipping_costs <= 950 
        THEN 1 ELSE 0 END) AS perfect_shipments,
    ROUND(100.0 * SUM(CASE 
        WHEN delivery_time_deviation <= 0 
        AND iot_temperature <= 15 
        AND shipping_costs <= 950 
        THEN 1 ELSE 0 END) / COUNT(*), 2) AS ftr_percentage
FROM {f_table}
GROUP BY supplier_id
HAVING total_shipments > 100
ORDER BY ftr_percentage DESC;
"""

print(f"--- 🏆 MASTER KPI: FIRST-TIME-RIGHT RATE ---")
ftr_results = sql(ftr_query)
display(ftr_results)

--- 🏆 MASTER KPI: FIRST-TIME-RIGHT RATE ---


,supplier_id,total_shipments,perfect_shipments,ftr_percentage
0,SUP_00358,2145,842.0,39.25
1,SUP_00568,1546,556.0,35.96
2,SUP_00143,2285,771.0,33.74
3,SUP_00429,3042,972.0,31.95
4,SUP_00959,3382,1076.0,31.82
...,...,...,...,...
1118,SUP_00875,2421,11.0,0.45
1119,SUP_00027,4777,19.0,0.40
1120,SUP_00548,1350,5.0,0.37
1121,SUP_00540,1397,4.0,0.29


In [29]:
# D. Supplier & Carrier Performance
# i> Supplier Reliability vs Actuals
# Purpose: Measures the gap between a supplier's rated reliability and their actual performance.
# Logic: Reliability Gap = (Actual Compliance % - Promised Reliability Score)

reliability_gap_query = f"""
WITH ActualPerformance AS (
    SELECT 
        supplier_id,
        COUNT(*) AS total_shipments,
        -- Actual Compliance: On-time (deviation <= 0) AND Thermal safe (<= 15)
        SUM(CASE WHEN delivery_time_deviation <= 0 AND iot_temperature <= 15 THEN 1 ELSE 0 END) AS compliant_shipments
    FROM {f_table}
    GROUP BY supplier_id
)
SELECT 
    s.supplier_id,
    s.supplier_reliability_score AS promised_reliability,
    ROUND(100.0 * a.compliant_shipments / a.total_shipments, 2) AS actual_reliability_pct,
    ROUND((100.0 * a.compliant_shipments / a.total_shipments) - s.supplier_reliability_score, 2) AS reliability_gap
FROM ActualPerformance a
JOIN dim_suppliers s ON a.supplier_id = s.supplier_id
WHERE a.total_shipments > 100
ORDER BY reliability_gap ASC -- Focus on suppliers failing to meet their 'promised' score
LIMIT 15;
"""

print(f"--- 🚚 ADVANCED KPI: RELIABILITY GAP ANALYSIS ---")
reliability_results = sql(reliability_gap_query)

# Display results
display(reliability_results)


--- 🚚 ADVANCED KPI: RELIABILITY GAP ANALYSIS ---


,supplier_id,promised_reliability,actual_reliability_pct,reliability_gap
0,SUP_00875,0.813262,0.50,-0.32
1,SUP_00540,0.499426,0.29,-0.21
2,SUP_00726,0.745368,0.55,-0.20
3,SUP_00863,0.849720,0.68,-0.16
4,SUP_00548,0.489505,0.74,0.25
5,SUP_00647,0.593935,1.03,0.43
6,SUP_00777,0.713987,1.35,0.63
7,SUP_00750,0.733388,1.47,0.74
8,SUP_00900,0.756950,1.75,1.00
9,SUP_00770,0.733760,2.08,1.35


In [30]:
# E. General
# i> Anomaly Detection
# Purpose: Identifies individual shipments that deviate significantly from the cost norm.
# Logic: Z-Score = (Current Cost - Avg Route Cost) / StdDev Route Cost

anomaly_detection_query = f"""
WITH RouteStats AS (
    SELECT 
        route_id,
        AVG(shipping_costs) AS avg_route_cost,
        STDDEV(shipping_costs) AS stddev_route_cost
    FROM {f_table}
    GROUP BY route_id
    HAVING COUNT(*) > 20 -- Statistical threshold
)
SELECT 
    f.Shipment_ID,
    f.supplier_id,
    f.route_id,
    f.shipping_costs,
    ROUND(r.avg_route_cost, 2) AS route_avg,
    ROUND((f.shipping_costs - r.avg_route_cost) / NULLIF(r.stddev_route_cost, 0), 2) AS z_score
FROM {f_table} f
JOIN RouteStats r ON f.route_id = r.route_id
WHERE (f.shipping_costs - r.avg_route_cost) / NULLIF(r.stddev_route_cost, 0) > 3
ORDER BY z_score DESC
LIMIT 20;
"""

print(f"--- 🚨 ADVANCED KPI: STATISTICAL ANOMALY DETECTION ---")
anomaly_results = sql(anomaly_detection_query)

# Display results
display(anomaly_results)

--- 🚨 ADVANCED KPI: STATISTICAL ANOMALY DETECTION ---


,Shipment_ID,supplier_id,route_id,shipping_costs,route_avg,z_score
0,1435696,SUP_00515,RTE_00760,984.675237,234.47,4.60
1,3307119,SUP_00515,RTE_00760,981.508211,234.47,4.59
2,2139049,SUP_00515,RTE_00760,981.874302,234.47,4.59
3,3322853,SUP_00515,RTE_00760,980.695225,234.47,4.58
4,2294798,SUP_00516,RTE_00760,979.468624,234.47,4.57
5,3772803,SUP_00504,RTE_00817,958.715636,243.30,4.24
6,3805215,SUP_00504,RTE_00817,957.857122,243.30,4.24
7,3238802,SUP_00504,RTE_00817,958.473794,243.30,4.24
8,2803129,SUP_00503,RTE_00817,958.043005,243.30,4.24
9,1297247,SUP_00504,RTE_00817,958.185417,243.30,4.24


In [ ]:
# F. Risk Management & Safety
# Driver Fatigue Correlation (Driver Workload vs. Performance Decay)
# Purpose: Detects if high workload correlates with increased "Red Flags".

fatigue_query = f"""
SELECT 
    f.driver_id,
    COUNT(f.Shipment_ID) AS shipment_count,
    ROUND(AVG(f.delivery_time_deviation), 2) AS avg_delay,
    ROUND(AVG(f.iot_temperature), 2) AS avg_temp,
    COUNT(CASE WHEN f.iot_temperature > 15 THEN 1 END) AS thermal_incidents
FROM {f_table} f
GROUP BY f.driver_id
HAVING shipment_count > 50
ORDER BY shipment_count DESC
LIMIT 15;
"""

print(f"--- ⚠️ RISK KPI: WORKLOAD VS. PERFORMANCE ---")
fatigue_results = sql(fatigue_query)
display(fatigue_results)

--- ⚠️ RISK KPI: WORKLOAD VS. PERFORMANCE ---


,driver_id,shipment_count,avg_delay,avg_temp,thermal_incidents
0,DRV_10295,82522,5.04,4.67,15432
1,DRV_10294,60848,5.14,4.44,10813
2,DRV_10293,30998,5.19,4.14,4947
3,DRV_10292,25430,4.64,4.19,4412
4,DRV_10290,21170,4.90,4.54,4319
5,DRV_10291,20574,4.39,4.91,4507
6,DRV_10289,20004,5.15,4.37,3611
7,DRV_00101,17744,5.07,5.37,4224
8,DRV_10288,15841,5.09,4.13,2629
9,DRV_00203,15142,4.88,4.87,3028


### Phase 4: Additional KPIs

**A. The "Thermal Dead-Zone" Query**

SUP_01123 has a 0% integrity score. We need to check if this is tied to specific routes.

In [3]:
# Identify if breaches are route-specific or vendor-wide
Thermal_Dead_Zone = f"""SELECT 
    supplier_id, 
    route_id, 
    COUNT(*) AS total_trips,
    SUM(CASE WHEN iot_temperature > 8 THEN 1 ELSE 0 END) AS breach_count,
    ROUND(AVG(iot_temperature), 2) AS avg_temp
FROM fact_shipments
WHERE supplier_id = 'SUP_01123'
GROUP BY 1, 2
ORDER BY breach_count DESC;"""


print(f"--- 🛰️ The Thermal Dead-Zone Query ---")
Results = sql(Thermal_Dead_Zone)

# Display the results
display(Results)

--- 🛰️ The Thermal Dead-Zone Query ---


,supplier_id,route_id,total_trips,breach_count,avg_temp
0,SUP_01123,RTE_01114,4792,0.0,NaN


**The "Reliability-Cost Paradox" Anomaly**

We have cost variance and reliability gap, but we need a script to find "The Worst Value Vendors" (High Variance + High Reliability Gap).

In [9]:
# Finding suppliers we pay more for, who deliver less
''' This combines Supplier performance with Shipment costs to 
find high-cost/low-reliability vendors (=supplier_performance_table).'''

Reliability_Cost_Paradox = f"""
WITH Performance_Metrics AS (
    SELECT 
        s.supplier_id,
        s.supplier_reliability_score AS promised_reliability,
        -- Calculate Actual Reliability (e.g., FTR rate or 1 - delay frequency)
        AVG(CASE WHEN f.delivery_time_deviation <= 0 THEN 1.0 ELSE 0.0 END) AS actual_reliability,
        -- Calculate Cost Variance against a global benchmark (using $462.30 from your previous data)
        AVG(f.shipping_costs) AS avg_cost,
        ((AVG(f.shipping_costs) - 462.30) / 462.30) * 100 AS variance_percentage
    FROM fact_shipments f
    JOIN dim_suppliers s ON f.supplier_id = s.supplier_id
    GROUP BY 1, 2
)
SELECT 
    supplier_id,
    ROUND(variance_percentage, 2) AS variance_pct,
    ROUND((actual_reliability - promised_reliability), 3) AS reliability_gap
FROM Performance_Metrics
WHERE variance_percentage > 20 
  AND (actual_reliability - promised_reliability) < -0.1
ORDER BY variance_percentage DESC;
"""

print(f"--- 🛰️ Reliability-Cost Paradox ---")
Results = sql(Reliability_Cost_Paradox)

# Display the results
display(Results)

--- 🛰️ Reliability-Cost Paradox ---


,supplier_id,variance_pct,reliability_gap
0,SUP_00484,47.64,-0.163
1,SUP_00858,42.93,-0.653
2,SUP_00583,37.11,-0.336
3,SUP_00546,35.07,-0.274
4,SUP_00664,31.45,-0.566
5,SUP_00578,30.79,-0.253
6,SUP_00562,29.91,-0.236
7,SUP_00773,28.44,-0.301
8,SUP_01086,27.32,-0.738
9,SUP_00579,26.99,-0.293


**C. Temporal "Drift" Anomaly**

Since we have a timestamp and iot_temperature, we should check if breaches are increasing over time (indicating fleet wear and tear).

In [10]:
## Month-over-Month Breach Growth
Month_over_Month_Breach_Growth = f"""SELECT 
    date_trunc('month', timestamp) AS month,
    COUNT(*) AS total_shipments,
    (SUM(CASE WHEN iot_temperature > 8 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS breach_rate
FROM fact_shipments
GROUP BY 1
ORDER BY 1;""" 

print(f"--- 🛰️ Month-over-Month Breach Growth ---")
Results = sql(Month_over_Month_Breach_Growth)

# Display the results
display(Results)

--- 🛰️ Month-over-Month Breach Growth ---


,month,total_shipments,breach_rate
0,2021-01-01,2392,23.996656
1,2021-02-01,4151,26.017827
2,2021-03-01,6807,24.944910
3,2021-04-01,8996,24.433081
4,2021-05-01,11590,24.918033
...,...,...,...
88,2028-05-01,9850,22.791878
89,2028-06-01,7315,22.788790
90,2028-07-01,5203,22.660004
91,2028-08-01,2824,25.070822


**D. Geospatial Risk**

To prepare the data for the Folium map without overloading it. It creates a "Heatmap Ready" dataset.

In [11]:
# Aggregating risks by Coordinate Bins (rounding to 1 decimal place ~11km precision)
Geospatial_Risk = f"""SELECT 
    ROUND(r.vehicle_gps_latitude, 1) AS lat_bin,
    ROUND(r.vehicle_gps_longitude, 1) AS lon_bin,
    COUNT(f.Shipment_ID) AS volume,
    AVG(f.delivery_time_deviation) AS avg_delay,
    SUM(CASE WHEN f.iot_temperature > 8 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS breach_rate_pct
FROM fact_shipments f
JOIN dim_routes r ON f.route_id = r.route_id
GROUP BY 1, 2
HAVING volume > 100;""" 
# Filter out low-volume noise


print(f"--- 🛰️ Geospatial Risk ---")
Results = sql(Geospatial_Risk)

# Display the results
display(Results)

--- 🛰️ Geospatial Risk ---


,lat_bin,lon_bin,volume,avg_delay,breach_rate_pct
0,33.5,-106.6,3119,5.454026,11.830715
1,30.1,-100.6,8964,4.459727,22.846943
2,30.3,-104.1,11669,5.273925,27.345959
3,45.4,-118.0,4684,4.737739,28.821520
4,47.7,-89.5,2327,5.603055,24.924796
...,...,...,...,...,...
1067,44.7,-84.3,2126,6.195516,28.927563
1068,40.9,-86.6,1300,4.029524,34.384615
1069,43.5,-93.8,1039,4.554178,21.077960
1070,44.6,-95.3,1601,2.886405,33.791380


### C. Closure of Connection:

In [13]:
con.close()